In [ ]:
import pandas as pd
from src.config.settings import settings
from pathlib import Path

In [ ]:
from pandas import DataFrame

stops_df: DataFrame = pd.read_csv(settings.staging_dir / "stops.csv")

stops_df = stops_df[["stop_id", "stop_name"]]
stops_df = stops_df[stops_df["stop_id"].str.startswith("StopArea:", na=False)]
stops_df = stops_df.rename(
    columns={"stop_id": "station_id", "stop_name": "station_name"}
)
stops_df["station_id"] = stops_df["station_id"].str.strip()
stops_df["station_name"] = stops_df["station_name"].str.strip().str.lower()
stops_df = (
    stops_df.dropna(subset=["station_id", "station_name"])
    .drop_duplicates(subset=["station_id"])
    .reset_index(drop=True)
)
station_dim_path = Path(settings.curated_dir / "dim_station.parquet")

# Mapping does not exist yet
if not station_dim_path.is_file():

    stops_df.insert(
        0,
        "station_key",
        range(1, len(stops_df) + 1),
    )

    settings.curated_dir.mkdir(parents=True, exist_ok=True)

    stops_df.to_parquet(
        station_dim_path,
        index=False,
    )
else:
    dim_station: DataFrame = pd.read_parquet(station_dim_path)

    new_stations: DataFrame = stops_df[
        ~stops_df["station_id"].isin(values=dim_station["station_id"])
    ].copy()

    next_key = 1
    if not dim_station.empty:
        next_key = int(dim_station["station_key"].max()) + 1

    new_stations.insert(
        0,
        "station_key",
        pd.Series(
            range(next_key, next_key + len(new_stations)),
            index=new_stations.index,
            dtype="Int64",
        ),
    )

    dim_station = pd.concat(
        [dim_station, new_stations],
        ignore_index=True,
    )

    dim_station = dim_station.set_index("station_id")

    dim_station.update(stops_df.set_index("station_id")[["station_name"]])

    dim_station = dim_station.reset_index()

    dim_station["station_key"] = dim_station["station_key"].astype("int64")

    dim_station.to_parquet(
        station_dim_path,
        index=False,
    )

In [ ]:
routes_df: DataFrame = pd.read_csv(
    settings.staging_dir / "routes.csv",
    dtype={
        "route_id": "string",
        "agency_id": "string",
        "route_short_name": "string",
        "route_long_name": "string",
        "route_type": "Int64",
    },
)

routes_df = routes_df[
    ["route_id", "agency_id", "route_short_name", "route_long_name", "route_type"]
].copy()

routes_df["route_id"] = routes_df["route_id"].str.strip()
routes_df["agency_id"] = routes_df["agency_id"].str.strip()
routes_df["route_short_name"] = routes_df["route_short_name"].str.strip().str.lower()
routes_df["route_long_name"] = routes_df["route_long_name"].str.strip().str.lower()

routes_df = (
    routes_df.dropna(subset=["route_id"])
    .drop_duplicates(subset=["route_id"])
    .reset_index(drop=True)
)

route_dim_path = Path(settings.curated_dir / "dim_routes.parquet")

if not route_dim_path.is_file():
    routes_df.insert(
        0,
        "route_key",
        range(1, len(routes_df) + 1),
    )

    dim_routes = routes_df.copy()

else:
    dim_routes: DataFrame = pd.read_parquet(route_dim_path)

    new_routes: DataFrame = routes_df[
        ~routes_df["route_id"].isin(dim_routes["route_id"])
    ].copy()

    next_key = int(dim_routes["route_key"].max()) + 1 if not dim_routes.empty else 1

    new_routes.insert(
        0,
        "route_key",
        range(next_key, next_key + len(new_routes)),
    )

    dim_routes = pd.concat(
        [dim_routes, new_routes],
        ignore_index=True,
    )

    dim_routes = dim_routes.set_index("route_id")

    dim_routes.update(routes_df.set_index("route_id"))

    dim_routes = dim_routes.reset_index()

dim_routes["route_key"] = dim_routes["route_key"].astype("int64")
dim_routes = dim_routes[
    [
        "route_key",
        "route_id",
        "agency_id",
        "route_short_name",
        "route_long_name",
        "route_type",
    ]
]

# Keep agency attributes in the route dimension for a star schema.
agency_df = pd.read_csv(settings.staging_dir / "agency.csv", dtype="string")
for column in agency_df.columns:
    agency_df[column] = agency_df[column].str.strip()
agency_df = agency_df.dropna(subset=["agency_id"]).drop_duplicates("agency_id")
dim_routes = dim_routes.merge(
    agency_df, on="agency_id", how="left", validate="many_to_one", indicator=True
)
if dim_routes["_merge"].ne("both").any():
    raise ValueError("Routes without a matching source agency")
dim_routes = dim_routes.drop(columns=["_merge"])

route_dim_path.parent.mkdir(parents=True, exist_ok=True)

dim_routes.to_parquet(
    route_dim_path,
    index=False,
)

In [ ]:
trips_df: DataFrame = pd.read_csv(
    settings.staging_dir / "trips.csv", dtype="string"
).drop(columns=["shape_id", "route_id"])

for column in trips_df.columns:
    trips_df[column] = trips_df[column].str.strip()

trips_df = (
    trips_df.dropna(subset=["trip_id"])
    .drop_duplicates(subset=["trip_id"])
    .reset_index(drop=True)
)
trip_dim_path = settings.curated_dir / "dim_trips.parquet"

if not trip_dim_path.is_file():
    dim_trips = trips_df.copy()
    dim_trips.insert(0, "trip_key", range(1, len(dim_trips) + 1))
else:
    dim_trips = pd.read_parquet(trip_dim_path)
    new_trips = trips_df[~trips_df["trip_id"].isin(dim_trips["trip_id"])].copy()
    next_key = int(dim_trips["trip_key"].max()) + 1 if not dim_trips.empty else 1
    new_trips.insert(0, "trip_key", range(next_key, next_key + len(new_trips)))
    dim_trips = pd.concat([dim_trips, new_trips], ignore_index=True)
    dim_trips = dim_trips.set_index("trip_id")
    dim_trips.update(trips_df.set_index("trip_id"))
    dim_trips = dim_trips.reset_index()

dim_trips["trip_key"] = dim_trips["trip_key"].astype("int64")
dim_trips = dim_trips[["trip_key", *trips_df.columns]]
trip_dim_path.parent.mkdir(parents=True, exist_ok=True)
dim_trips.to_parquet(trip_dim_path, index=False)


In [ ]:
stop_times_df: DataFrame = pd.read_csv(
    settings.staging_dir / "stop_times.csv", dtype="string"
).drop(columns=["shape_dist_traveled"])
for column in stop_times_df.columns:
    stop_times_df[column] = stop_times_df[column].str.strip()

# Load the persisted dimension mappings after all dimensions have been built.
trip_keys = pd.read_parquet(
    settings.curated_dir / "dim_trips.parquet",
    columns=["trip_id", "trip_key"],
)
# Route membership is a staging mapping, not a dimension-to-dimension link.
trip_routes = pd.read_csv(
    settings.staging_dir / "trips.csv", usecols=["trip_id", "route_id"], dtype="string"
)
for column in trip_routes.columns:
    trip_routes[column] = trip_routes[column].str.strip()
trip_routes = trip_routes.dropna(subset=["trip_id"]).drop_duplicates("trip_id")
trip_keys = trip_keys.merge(
    trip_routes, on="trip_id", how="left", validate="one_to_one"
)
route_keys = pd.read_parquet(
    settings.curated_dir / "dim_routes.parquet", columns=["route_id", "route_key"]
)
station_keys = pd.read_parquet(
    settings.curated_dir / "dim_station.parquet", columns=["station_id", "station_key"]
)
stop_station_map = pd.read_csv(
    settings.staging_dir / "stops.csv",
    usecols=["stop_id", "parent_station"], dtype="string",
)
for column in stop_station_map.columns:
    stop_station_map[column] = stop_station_map[column].str.strip()
stop_station_map["station_id"] = stop_station_map["parent_station"].mask(
    stop_station_map["parent_station"].isna()
    | stop_station_map["parent_station"].eq(""),
    stop_station_map["stop_id"],
)
stop_station_map = stop_station_map.merge(
    station_keys, on="station_id", how="left", validate="many_to_one"
)
fact_stop_times = (
    stop_times_df.merge(trip_keys, on="trip_id", how="left", validate="many_to_one")
    .merge(route_keys, on="route_id", how="left", validate="many_to_one")
    .merge(
        stop_station_map[["stop_id", "station_key"]],
        on="stop_id", how="left", validate="many_to_one",
    )
)
for key in ["trip_key", "route_key", "station_key"]:
    if fact_stop_times[key].isna().any():
        raise ValueError(f"Unresolved {key} for {fact_stop_times[key].isna().sum()} stop visits")
    fact_stop_times[key] = fact_stop_times[key].astype("int64")

fact_stop_times = fact_stop_times[
    ["trip_key", "route_key", "station_key", "arrival_time", "departure_time",
     "stop_sequence", "stop_headsign", "pickup_type", "drop_off_type"]
]
stop_times_fact_path = settings.curated_dir / "fact_stop_times.parquet"
stop_times_fact_path.parent.mkdir(parents=True, exist_ok=True)
fact_stop_times.to_parquet(stop_times_fact_path, index=False)
